### Language copies (oxjob #2186, step 5b-F of CreateWorkAuthorships)

Some repositories and journals print every author again in another language: Ukrainian, Russian, English (romanized), full name and initials. Each copy becomes a seat of its own, often on a profile of its own. This task finds bylines that are entirely language copies of the same people in the same order and writes, per seat, the seat it folds into. CreateWorkAuthorships step 5b-F drops the copies after author ids bind (affiliations, ORCID and institutions merge into the kept seat; the kept seat takes the person's first non-empty author id).

Source of truth and tests for the UDF: oxjobs #2186 `work/ukru_rule/` (`names.py`, `layout.py`, `build_udf_sql.py`, `test_fixtures.py`). Needs only `openalex_works_base`; runs after Works_Base, off the critical path.


In [0]:
-- oxjob #2186 step 5b-F: per seat, the seat it folds into when the whole byline is language copies of the same people
-- in the same order (Ukrainian / Russian / Belarusian / romanized copies, blocks or per-person groups), else NULL.
-- GENERATED from oxjobs #2186 work/ukru_rule/names.py + layout.py by build_udf_sql.py (source of truth and tests there).
-- Revert: drop this function and the Language_Copies task.
CREATE OR REPLACE FUNCTION openalex.works.language_copy_targets(names ARRAY<STRING>, orcids ARRAY<STRING>)
RETURNS ARRAY<INT>
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #2186 step 5b-F: target seat per seat when the byline is language copies (else NULL). Source + tests: oxjobs #2186 work/ukru_rule'
AS $$
import collections

import itertools

import re

import sys

import unicodedata



MAP_DATA = {"абзалович": ["абзалович"], "абрамивна": ["абрамовна"], "абрамович": ["абрамович"], "абрамовна": ["абрамивна"], "августивна": ["августовна"], "августовна": ["августивна"], "авдеевич": ["авдииович"], "авдииович": ["авдеевич"], "авенирович": ["авенирович"], "авксентевич": ["авксентииович"], "авксентевна": ["авксентиивна"], "авксентиивна": ["авксентевна"], "авксентииович": ["авксентевич"], "адам": ["адам"], "адамович": ["адамович"], "аделаида": ["аделаида"], "азиза": ["азиза"], "аикануш": ["аикануш"], "акимовна": ["якимивна"], "алберт": ["алберт"], "албертивна": ["албертовна"], "албертович": ["албертович"], "албертовна": ["албертивна"], "албина": ["албина"], "албинович": ["албинович"], "алевтина": ["алевтина"], "александр": ["олександр"], "александра": ["олександра"], "александрович": ["олександрович"], "александровна": ["олександривна"], "алексеевич": ["олексииович"], "алексеевна": ["олексиивна"], "алексеи": ["олексии"], "алена": ["алона"], "алеся": ["алеся"], "аликович": ["аликович"], "аликовна": ["аликовна"], "алим": ["алим"], "алима": ["алима"], "алимович": ["алимович"], "алина": ["алина"], "алиса": ["алиса"], "алия": ["алия"], "алла": ["алла"], "алона": ["алена"], "аминет": ["аминет"], "амфиан": ["амфиан"], "анаит": ["анаит"], "анастасия": ["анастасия"], "анатолевич": ["анатолевич", "анатолиевич", "анатолииович"], "анатолевна": ["анатолиивна"], "анатолиевич": ["анатолевич"], "анатолии": ["анатолии"], "анатолиивна": ["анатолевна"], "анатолииович": ["анатолевич"], "ангелина": ["ангелина"], "андреевич": ["андрииович"], "андреевна": ["андриивна"], "андреи": ["андреи", "андрии"], "андреянна": ["андреянна"], "андриана": ["андриана"], "андрианович": ["андрианович"], "андрии": ["андреи"], "андриивна": ["андреевна"], "андрииович": ["андреевич"], "анжела": ["анжела"], "анжелика": ["анжелика"], "анжелина": ["анжелина"], "анна": ["анна", "ганна"], "аннета": ["аннета"], "анрис": ["анрис"], "антон": ["антон"], "антонавич": ["антонович"], "антонивна": ["антоновна"], "антонина": ["антонина"], "антонович": ["антонавич", "антонович"], "антоновна": ["антонивна"], "ануш": ["ануш"], "аншелевич": ["аншелевич"], "аншелович": ["аншелович"], "арвидович": ["арвидович"], "арина": ["арина"], "аркадевич": ["аркадииович"], "аркадевна": ["аркадиивна"], "аркадиевич": ["аркадииович"], "аркадии": ["аркадии"], "аркадиивна": ["аркадевна"], "аркадииович": ["аркадевич", "аркадиевич"], "арнолд": ["арнолд"], "арнолдович": ["арнолдович"], "арсен": ["арсен"], "артем": ["артем"], "артемович": ["артемович"], "артур": ["артур"], "артуривна": ["артуровна"], "артурович": ["артурович"], "артуровна": ["артуривна"], "асирович": ["асирович"], "асият": ["асият"], "афанасевич": ["афанасевич", "афанасииович"], "афанасевна": ["афанасиивна"], "афанасиивна": ["афанасевна"], "афанасииович": ["афанасевич"], "богдан": ["богдан"], "богдана": ["богдана"], "богданивна": ["богдановна"], "богданович": ["богданович"], "богдановна": ["богданивна"], "борис": ["борис"], "борисивна": ["борисовна"], "бориславивна": ["бориславовна"], "бориславовна": ["бориславивна"], "борисович": ["борисович"], "борисовна": ["борисивна"], "бронислав": ["бронислав"], "брониславивна": ["брониславовна"], "брониславович": ["брониславович"], "брониславовна": ["брониславивна"], "ваагн": ["ваагн"], "вадим": ["вадим"], "вадимивна": ["вадимовна"], "вадимович": ["вадимович"], "вадимовна": ["вадимивна"], "валдемар": ["валдемар"], "валентин": ["валентин"], "валентина": ["валентина"], "валентинивна": ["валентиновна"], "валентинович": ["валентинович"], "валентиновна": ["валентинивна"], "валеревич": ["валерииович"], "валеревна": ["валериивна"], "валериан": ["валериан"], "валерианивна": ["валериановна"], "валериановна": ["валерианивна"], "валериевич": ["валерииович"], "валериевна": ["валериивна"], "валерии": ["валерии"], "валериивна": ["валеревна", "валериевна"], "валерииович": ["валеревич", "валериевич"], "валерия": ["валерия"], "ванда": ["ванда"], "варвара": ["варвара"], "васил": ["василии"], "василевич": ["василевич", "василович"], "василевна": ["василивна", "василиивна"], "василивна": ["василевна"], "василии": ["васил", "василии"], "василиивна": ["василевна"], "василина": ["василиса"], "василиса": ["василина"], "василович": ["василевич"], "вениамин": ["вениамин"], "вениаминивна": ["вениаминовна"], "вениаминович": ["вениаминович"], "вениаминовна": ["вениаминивна"], "вера": ["вира"], "вероника": ["вероника"], "видаддиевич": ["видаддиевич"], "виктор": ["виктор"], "викторивна": ["викторовна"], "виктория": ["виктория"], "викторович": ["викторович"], "викторовна": ["викторивна", "викторовна"], "вилгелмович": ["вилгелмович"], "вилена": ["вилена"], "вилям": ["вилям"], "виолета": ["виолета"], "вира": ["вера"], "вита": ["вита"], "виталевич": ["виталииович"], "виталевна": ["виталивна", "виталиивна"], "виталивна": ["виталевна"], "виталии": ["виталии"], "виталиивна": ["виталевна"], "виталииович": ["виталевич"], "виталина": ["виталина"], "виталия": ["виталия"], "витолдович": ["витолдович"], "влада": ["влада"], "владиленович": ["владиленович"], "владимир": ["володимир", "уладзимир"], "владимирович": ["володимирович", "уладзимиравич"], "владимировна": ["володимиривна"], "владислав": ["владислав"], "владислава": ["владислава"], "владиславивна": ["владиславовна"], "владиславович": ["владиславович"], "владиславовна": ["владиславивна"], "владлена": ["владлена"], "владленивна": ["владленовна"], "владленовна": ["владленивна"], "власович": ["власович"], "володимир": ["владимир"], "володимиривна": ["владимировна"], "володимирович": ["владимирович"], "всеволод": ["всеволод"], "всеволодивна": ["всеволодовна"], "всеволодович": ["всеволодович"], "всеволодовна": ["всеволодивна"], "вячеслав": ["вячеслав"], "вячеславивна": ["вячеславовна"], "вячеславович": ["вячеславович"], "вячеславовна": ["вячеславивна"], "гавриливна": ["гавриловна"], "гаврилович": ["гаврилович"], "гавриловна": ["гавриливна"], "галина": ["галина"], "галия": ["галия"], "ганна": ["анна"], "гариславович": ["гариславович"], "гарри": ["гарри"], "геворгивна": ["геворговна"], "геворговна": ["геворгивна"], "гелфира": ["гелфира"], "генефа": ["генефа"], "геннадевич": ["геннадииович"], "геннадевна": ["геннадиивна"], "геннадиевич": ["геннадииович"], "геннадиевна": ["геннадиивна"], "геннадии": ["геннадии"], "геннадиивна": ["геннадевна", "геннадиевна"], "геннадииович": ["геннадевич", "геннадиевич"], "генрих": ["генрих"], "генрихович": ["генрихович"], "георгиевич": ["георгиевич", "георгииович"], "георгиевна": ["георгиивна"], "георгии": ["георгии"], "георгиивна": ["георгиевна"], "георгииович": ["георгиевич"], "герман": ["герман"], "германивна": ["германовна"], "германович": ["германович"], "германовна": ["германивна"], "глеб": ["глеб", "глиб"], "глебович": ["глибович"], "глиб": ["глеб"], "глибович": ["глебович"], "гордеевич": ["гордииович"], "гордииович": ["гордеевич"], "григоревич": ["григорииович", "григорович"], "григоревна": ["григоривна", "григориивна"], "григоривна": ["григоревна"], "григории": ["григории"], "григориивна": ["григоревна"], "григорииович": ["григоревич"], "григорович": ["григоревич"], "давидивна": ["давидовна"], "давидовна": ["давидивна"], "даниил": ["данило"], "даниливна": ["даниловна"], "данило": ["даниил", "данило"], "данилович": ["данилович"], "даниловна": ["даниливна"], "дария": ["дария", "даря"], "даря": ["дария"], "денис": ["денис"], "денисивна": ["денисовна"], "денисович": ["денисович"], "денисовна": ["денисивна"], "джамашевич": ["джамашевич"], "джонивна": ["джоновна"], "джоновна": ["джонивна"], "диана": ["диана"], "дина": ["дина"], "дмитривна": ["дмитриевна"], "дмитриевич": ["дмитрович"], "дмитриевна": ["дмитривна", "дмитриивна"], "дмитрии": ["дмитрии", "дмитро"], "дмитриивна": ["дмитриевна"], "дмитро": ["дмитрии"], "дмитрович": ["дмитриевич"], "евген": ["евгении"], "евгеневич": ["евгениевич", "евгенииович", "евгенович"], "евгеневна": ["евгенивна", "евгениивна"], "евгенивна": ["евгеневна"], "евгениевич": ["евгеневич"], "евгении": ["евген", "евгении"], "евгениивна": ["евгеневна"], "евгенииович": ["евгеневич"], "евгения": ["евгения"], "евгенович": ["евгеневич"], "евграфович": ["евграфович"], "евдокия": ["евдокия"], "евелина": ["евелина"], "евстафевич": ["евстафииович"], "евстафевна": ["евстафиивна"], "евстафиивна": ["евстафевна"], "евстафииович": ["евстафевич"], "егор": ["егор"], "егоривна": ["егоровна"], "егорович": ["егорович"], "егоровна": ["егоривна"], "едуард": ["едуард"], "едуардивна": ["едуардовна"], "едуардович": ["едуардович"], "едуардовна": ["едуардивна"], "екатерина": ["катерина"], "елвина": ["елвина"], "елвира": ["елвира"], "елена": ["елена", "олена"], "елеонора": ["елеонора"], "елзара": ["елзара"], "елизавета": ["елизавета"], "елина": ["елина"], "елла": ["елла"], "еллана": ["еллана"], "емануилович": ["еммануилович"], "емеляновна": ["омелянивна"], "емилевич": ["емилевич", "емилович"], "емилия": ["емилия"], "емилович": ["емилевич"], "емма": ["емма"], "еммануилович": ["емануилович", "еммануилович"], "ервинович": ["ервинович"], "ерикивна": ["ериковна"], "ериковна": ["ерикивна"], "ернестивна": ["ернестовна"], "ернестовна": ["ернестивна"], "ерофеи": ["ерофеи"], "ефимович": ["юхимович"], "ефимовна": ["юхимивна"], "ефрем": ["ефрем"], "ефремович": ["ефремович"], "жанетта": ["жанетта"], "жанна": ["жанна"], "жаннета": ["жаннета"], "захар": ["захар"], "захарии": ["захарии"], "захарович": ["захарович"], "зинаида": ["зинаида"], "зиновевич": ["зиновииович"], "зиновевна": ["зиновиивна"], "зиновии": ["зиновии"], "зиновиивна": ["зиновевна"], "зиновииович": ["зиновевич"], "златослава": ["златослава"], "зоряна": ["зоряна"], "зоя": ["зоя"], "ибрагимович": ["ибрагимович"], "иван": ["иван"], "иванавич": ["иванович"], "иванивна": ["ивановна"], "иванна": ["иванна"], "иванович": ["иванавич", "иванович"], "ивановна": ["иванивна"], "игор": ["игор"], "игоревич": ["игоревич", "игорович"], "игоревна": ["игоревна", "игоривна"], "игоривна": ["игоревна"], "игорович": ["игоревич"], "изабелла": ["изабелла"], "измаил": ["измаил"], "иларионивна": ["илларионовна"], "иларионович": ["илларионович"], "илинична": ["илливна"], "илич": ["иллич"], "илкович": ["илкович"], "илларионович": ["иларионович"], "илларионовна": ["иларионивна"], "илливна": ["илинична"], "иллич": ["илич"], "илля": ["иля"], "илона": ["илона"], "иля": ["илля"], "инга": ["инга"], "инеса": ["инесса"], "инесса": ["инеса", "инесса"], "инна": ["инна"], "иович": ["иович"], "ион": ["ион"], "иосип": ["иосиф"], "иосипивна": ["иосифовна"], "иосипович": ["иосифович"], "иосиф": ["иосип"], "иосифович": ["иосипович"], "иосифовна": ["иосипивна"], "ираида": ["ираида"], "ирена": ["ирена"], "ирина": ["ирина"], "иринарх": ["иринарх"], "исаакович": ["исакович"], "исааковна": ["исакивна"], "исакивна": ["исааковна"], "исакович": ["исаакович"], "ия": ["ия"], "камилович": ["камилович"], "капитолина": ["капитолина"], "каренович": ["каренович"], "карина": ["карина"], "карине": ["карине"], "катерина": ["екатерина"], "киазович": ["киазович"], "ким": ["ким"], "киндратович": ["кондратевич"], "кира": ["кира"], "кириливна": ["кирилловна"], "кирилл": ["кирило"], "кириллович": ["кирилович"], "кирилловна": ["кириливна"], "кирило": ["кирилл"], "кирилович": ["кириллович"], "клавдия": ["клавдия"], "клара": ["клара"], "климивна": ["климовна"], "климовна": ["климивна"], "кондратевич": ["киндратович"], "константин": ["костянтин"], "константинивна": ["константиновна"], "константинович": ["константинович", "костянтинович"], "константиновна": ["константинивна", "костянтинивна"], "костянтин": ["константин"], "костянтинивна": ["константиновна"], "костянтинович": ["константинович"], "кристина": ["кристина", "христина"], "ксения": ["ксения"], "кузмивна": ["кузминична"], "кузминична": ["кузмивна"], "кузмич": ["кузмич"], "курбанбаиович": ["курбанбаиович"], "лаврентевич": ["лаврентииович"], "лаврентевна": ["лаврентиивна"], "лаврентиивна": ["лаврентевна"], "лаврентииович": ["лаврентевич"], "лада": ["лада"], "лали": ["лали"], "лариса": ["лариса"], "лвивна": ["лвовна"], "лвович": ["лвович"], "лвовна": ["лвивна"], "лев": ["лев"], "леонард": ["леонард"], "леонивна": ["леоновна"], "леонид": ["леонид"], "леонидивна": ["леонидовна"], "леонидович": ["леонидович"], "леонидовна": ["леонидивна"], "леоновна": ["леонивна"], "леонтевич": ["леонтииович"], "леонтевна": ["леонтиивна"], "леонтиевич": ["леонтииович"], "леонтиевна": ["леонтиивна"], "леонтии": ["леонтии"], "леонтиивна": ["леонтевна", "леонтиевна"], "леонтииович": ["леонтевич", "леонтиевич"], "леополдович": ["леополдович"], "леся": ["леся"], "лиана": ["лиана"], "лидия": ["лидия"], "лилиана": ["лилиана"], "лилита": ["лилита"], "лилия": ["лилия"], "лиляна": ["лиляна"], "лина": ["лина"], "луарсабович": ["луарсабович"], "луиза": ["луиза"], "лука": ["лука"], "лукивна": ["лукинична"], "лукинична": ["лукивна"], "лукич": ["лукич"], "любава": ["любава"], "любов": ["любов"], "любомира": ["любомира"], "любомиривна": ["любомировна"], "любомирович": ["любомирович"], "любомировна": ["любомиривна"], "любославович": ["любославович"], "людвигович": ["людвигович"], "людмила": ["людмила"], "маия": ["маия"], "макаривна": ["макаровна"], "макарович": ["макарович"], "макаровна": ["макаривна"], "максим": ["максим"], "максимивна": ["максимовна"], "максимович": ["максимович"], "максимовна": ["максимивна"], "малвина": ["малвина"], "маликивна": ["маликовна"], "маликовна": ["маликивна"], "маргарита": ["маргарита"], "марианна": ["марианна"], "марина": ["марина"], "мария": ["мария"], "марк": ["марк", "марко"], "маркивна": ["марковна"], "марко": ["марк"], "маркович": ["маркович"], "марковна": ["маркивна", "марковна"], "марта": ["марта"], "мартинович": ["мартинович"], "маруша": ["маруша"], "матвеевич": ["матвииович"], "матвеи": ["матвии"], "матвии": ["матвеи"], "матвииович": ["матвеевич"], "мечиславович": ["мечиславович"], "микита": ["никита"], "микитович": ["никитович"], "микола": ["николаи"], "миколаевич": ["николаевич"], "миколаи": ["николаи"], "миколаивна": ["николаевна"], "миколаиович": ["николаевич"], "минас": ["минас"], "мирон": ["мирон"], "миронивна": ["мироновна"], "миронович": ["миронович"], "мироновна": ["миронивна"], "мирослав": ["мирослав"], "мирослава": ["мирослава"], "мирославивна": ["мирославовна"], "мирославович": ["мирославович"], "мирославовна": ["мирославивна"], "митрофан": ["митрофан"], "митрофанивна": ["митрофановна"], "митрофанович": ["митрофанович"], "митрофановна": ["митрофанивна"], "михаил": ["михаило"], "михаиливна": ["михаиловна"], "михаило": ["михаил"], "михаилович": ["михаилович"], "михаиловна": ["михаиливна"], "набиевич": ["набиевич"], "надежда": ["надия"], "надиря": ["надиря"], "надия": ["надежда"], "назар": ["назар"], "назарович": ["назарович"], "назия": ["назия"], "нана": ["нана"], "наталия": ["наталия", "наталя"], "наталя": ["наталия", "наталя"], "наум": ["наум"], "наумивна": ["наумовна"], "наумович": ["наумович"], "наумовна": ["наумивна"], "нектар": ["нектар"], "нелли": ["нелли"], "нелля": ["нелля"], "неля": ["неля"], "несторович": ["несторович"], "никита": ["микита"], "никитович": ["микитович"], "никифор": ["никифор"], "никифоривна": ["никифоровна"], "никифорович": ["никифорович"], "никифоровна": ["никифоривна"], "николаевич": ["миколаевич", "миколаиович"], "николаевна": ["миколаивна"], "николаи": ["микола", "миколаи"], "нина": ["нина"], "нинел": ["нинел"], "ниязи": ["ниязи"], "нодарович": ["нодарович"], "нонна": ["нонна"], "оксана": ["оксана"], "октябрина": ["октябрина"], "олга": ["олга"], "олег": ["олег"], "олегивна": ["олеговна"], "олегович": ["олегович"], "олеговна": ["олегивна"], "олександр": ["александр"], "олександра": ["александра"], "олександривна": ["александровна"], "олександрович": ["александрович"], "олексии": ["алексеи"], "олексиивна": ["алексеевна"], "олексииович": ["алексеевич"], "олена": ["елена"], "олеся": ["олеся"], "омелянивна": ["емеляновна"], "опанасивна": ["опанасовна"], "опанасовна": ["опанасивна"], "орест": ["орест"], "ореста": ["ореста"], "орестович": ["орестович"], "оскаривна": ["оскаровна"], "оскаровна": ["оскаривна"], "остапович": ["остапович"], "павел": ["павел", "павло"], "павливна": ["павловна"], "павло": ["павел"], "павлович": ["павлович"], "павловна": ["павливна"], "пантелеимонивна": ["пантелеимоновна"], "пантелеимоновна": ["пантелеимонивна"], "петр": ["петро"], "петривна": ["петровна"], "петро": ["петр"], "петрович": ["петрович"], "петровна": ["петривна"], "пилипивна": ["филипповна"], "пилипович": ["филиппович"], "пинхусович": ["пинхусович"], "полина": ["полина"], "прокопевна": ["прокопивна"], "прокопивна": ["прокопевна"], "равил": ["равил"], "раиса": ["раиса"], "рамзесович": ["рамзесович"], "рашид": ["рашид"], "рашидивна": ["рашидовна"], "рашидовна": ["рашидивна"], "ревазивна": ["ревазовна"], "ревазовна": ["ревазивна"], "регина": ["регина"], "рената": ["рената"], "ризван": ["ризван"], "римма": ["римма"], "ричард": ["ричард"], "робертивна": ["робертовна"], "робертовна": ["робертивна"], "родион": ["родион"], "родионович": ["родионович"], "роза": ["роза"], "роксолана": ["роксолана"], "роман": ["роман"], "романивна": ["романовна"], "романович": ["романович"], "романовна": ["романивна"], "ростислав": ["ростислав"], "ростиславивна": ["ростиславовна"], "ростиславович": ["ростиславович"], "ростиславовна": ["ростиславивна"], "рувимович": ["рувимович"], "рудолфивна": ["рудолфовна"], "рудолфовна": ["рудолфивна"], "руслан": ["руслан"], "руслана": ["руслана"], "русланивна": ["руслановна"], "русланович": ["русланович"], "руслановна": ["русланивна"], "сабина": ["сабина"], "савелевич": ["савелииович"], "савелевна": ["савелиивна"], "савелиевна": ["савелиивна"], "савелиивна": ["савелевна", "савелиевна"], "савелииович": ["савелевич"], "савич": ["савич"], "самира": ["самира"], "самоилович": ["самоилович"], "самуил": ["самуил"], "самуилович": ["самуилович"], "самуиловна": ["самуиловна"], "сахнович": ["сахнович"], "светлана": ["свитлана"], "свитлана": ["светлана"], "святослав": ["святослав"], "святославивна": ["святославовна"], "святославович": ["святославович"], "святославовна": ["святославивна"], "семен": ["семен"], "семенивна": ["семеновна"], "семенович": ["семенович"], "семеновна": ["семенивна"], "сергеевич": ["сергииович"], "сергеевна": ["сергиивна"], "сергеи": ["сергии"], "сергии": ["сергеи"], "сергиивна": ["сергеевна"], "сергииович": ["сергеевич"], "сибилла": ["сибилла"], "силвестрович": ["силвестрович"], "симеонивна": ["симеоновна"], "симеоновна": ["симеонивна"], "снежана": ["снежана", "снижана"], "снижана": ["снежана"], "соломонович": ["соломонович"], "софия": ["софия"], "станислав": ["станислав"], "станислава": ["станислава"], "станиславивна": ["станиславовна"], "станиславович": ["станиславович"], "станиславовна": ["станиславивна"], "стелла": ["стелла"], "степан": ["степан"], "степанивна": ["степановна"], "степания": ["степания"], "степанович": ["степанович"], "степановна": ["степанивна"], "стефан": ["стефан"], "стефания": ["стефания"], "сулико": ["сулико"], "сусанна": ["сусанна"], "таиса": ["таиса"], "таисия": ["таисия"], "тамара": ["тамара"], "тамила": ["тамила"], "тарас": ["тарас"], "тарасивна": ["тарасовна"], "тарасовна": ["тарасивна"], "татяна": ["тетяна"], "теодорович": ["теодорович"], "терентевна": ["терентиивна"], "терентиивна": ["терентевна"], "тетяна": ["татяна"], "тимофеевич": ["тимофииович"], "тимофеевна": ["тимофиивна"], "тимофеи": ["тимофии"], "тимофии": ["тимофеи"], "тимофиивна": ["тимофеевна"], "тимофииович": ["тимофеевич"], "тимур": ["тимур"], "тихонович": ["тихонович"], "трифонивна": ["трифоновна"], "трифонович": ["трифонович"], "трифоновна": ["трифонивна"], "трофимович": ["трохимович"], "трофимовна": ["трохимивна"], "трохимивна": ["трофимовна"], "трохимович": ["трофимович"], "уладзимир": ["владимир"], "уладзимиравич": ["владимирович"], "уляна": ["уляна"], "улянович": ["улянович"], "фаина": ["фаина"], "федир": ["федор"], "федор": ["федир"], "федоривна": ["федоровна"], "федорович": ["федорович"], "федоровна": ["федоривна"], "феликс": ["феликс"], "феликсивна": ["феликсовна"], "феликсович": ["феликсович"], "феликсовна": ["феликсивна"], "феодосиевна": ["феодосиивна"], "феодосиивна": ["феодосиевна"], "филиппович": ["пилипович"], "филипповна": ["пилипивна"], "фомич": ["фомич"], "францевич": ["францевич", "францович"], "францевна": ["францивна"], "францивна": ["францевна", "францовна"], "францович": ["францевич"], "францовна": ["францивна"], "хашматулла": ["хашматулла"], "христина": ["кристина"], "чеслав": ["чеслав"], "шабанивна": ["шабановна"], "шабановна": ["шабанивна"], "шамил": ["шамил"], "шамилевич": ["шамилович"], "шамилович": ["шамилевич"], "юлевич": ["юлииович"], "юлевна": ["юлиивна"], "юлиан": ["юлиан"], "юлиана": ["юлиана"], "юлиевна": ["юлиивна"], "юлии": ["юлии"], "юлиивна": ["юлевна", "юлиевна"], "юлииович": ["юлевич"], "юлия": ["юлия"], "юревич": ["юриевич", "юрииович"], "юревна": ["юриивна"], "юриевич": ["юревич"], "юрии": ["юрии"], "юриивна": ["юревна"], "юрииович": ["юревич"], "юхимивна": ["ефимовна"], "юхимович": ["ефимович"], "ягубович": ["ягубович"], "якив": ["яков"], "якивна": ["яковлевна"], "якимивна": ["акимовна"], "яков": ["якив"], "якович": ["яковлевич"], "яковлевич": ["якович", "яковлевич"], "яковлевна": ["якивна"], "ян": ["ян"], "яна": ["яна"], "янивна": ["яновна"], "янина": ["янина"], "янович": ["янович"], "яновна": ["янивна"], "ярина": ["ярина"], "ярослав": ["ярослав"], "ярослава": ["ярослава"], "ярославивна": ["ярославовна"], "ярославна": ["ярославна"], "ярославович": ["ярославович"], "ярославовна": ["ярославивна"], "ясмина": ["ясмина"]}

RAW_DATA = {"абзалович": ["абзалович"], "абрамивна": ["абрамовна"], "абрамович": ["абрамович"], "абрамовна": ["абрамівна"], "августивна": ["августовна"], "августовна": ["августівна"], "авдеевич": ["авдійович"], "авдииович": ["авдеевич"], "авенирович": ["авенирович", "авенірович"], "авксентевич": ["авксентійович"], "авксентевна": ["авксентіївна"], "авксентиивна": ["авксентьевна"], "авксентииович": ["авксентьевич"], "адам": ["адам"], "адамович": ["адамович"], "аделаида": ["аделаида", "аделаїда"], "азиза": ["азиза", "азіза"], "аикануш": ["айкануш"], "акимовна": ["якимівна"], "алберт": ["альберт"], "албертивна": ["альбертовна"], "албертович": ["альбертович"], "албертовна": ["альбертівна"], "албина": ["альбина", "альбіна"], "албинович": ["альбинович", "альбінович"], "алевтина": ["алевтина", "алевтіна"], "александр": ["олександр"], "александра": ["олександра"], "александрович": ["олександрович"], "александровна": ["олександрівна"], "алексеевич": ["олексійович"], "алексеевна": ["олексіївна"], "алексеи": ["олексій"], "алена": ["альона"], "алеся": ["алеся"], "аликович": ["аликович", "алікович"], "аликовна": ["аликовна", "аліковна"], "алим": ["алим", "алім"], "алима": ["алима", "аліма"], "алимович": ["алимович", "алімович"], "алина": ["алина", "аліна"], "алиса": ["алиса", "аліса"], "алия": ["алия", "алія"], "алла": ["алла"], "алона": ["алена"], "аминет": ["аминет", "амінет"], "амфиан": ["амфиан", "амфіан"], "анаит": ["анаит", "анаіт"], "анастасия": ["анастасия", "анастасія"], "анатолевич": ["анатольевич", "анатольевіч", "анатолійович", "анатолієвич"], "анатолевна": ["анатоліївна"], "анатолиевич": ["анатольевич"], "анатолии": ["анатолий", "анатолій"], "анатолиивна": ["анатольевна"], "анатолииович": ["анатольевич"], "ангелина": ["ангелина", "ангеліна"], "андреевич": ["андрійович"], "андреевна": ["андріївна"], "андреи": ["андрей", "андрэй", "андрій"], "андреянна": ["андреянна"], "андриана": ["андриана", "андріана"], "андрианович": ["андрианович", "андріанович"], "андрии": ["андрей"], "андриивна": ["андреевна"], "андрииович": ["андреевич"], "анжела": ["анжела"], "анжелика": ["анжелика", "анжеліка"], "анжелина": ["анжелина", "анжеліна"], "анна": ["анна", "ганна"], "аннета": ["аннета"], "анрис": ["анрис", "анріс"], "антон": ["антон"], "антонавич": ["антонович"], "антонивна": ["антоновна"], "антонина": ["антонина", "антоніна"], "антонович": ["антонавіч", "антонович"], "антоновна": ["антонівна"], "ануш": ["ануш"], "аншелевич": ["аншелевич", "аншелєвич"], "аншелович": ["аншелович"], "арвидович": ["арвидович", "арвідович"], "арина": ["арина"], "аркадевич": ["аркадійович"], "аркадевна": ["аркадіївна"], "аркадиевич": ["аркадійович"], "аркадии": ["аркадий", "аркадій"], "аркадиивна": ["аркадьевна"], "аркадииович": ["аркадиевич", "аркадьевич"], "арнолд": ["арнольд"], "арнолдович": ["арнольдович"], "арсен": ["арсен"], "артем": ["артем"], "артемович": ["артемович"], "артур": ["артур"], "артуривна": ["артуровна"], "артурович": ["артурович"], "артуровна": ["артурівна"], "асирович": ["асирович", "асірович"], "асият": ["асият", "асіят"], "афанасевич": ["афанасьевич", "афанасьевіч", "афанасійович"], "афанасевна": ["афанасіївна"], "афанасиивна": ["афанасьевна"], "афанасииович": ["афанасьевич"], "богдан": ["богдан"], "богдана": ["богдана"], "богданивна": ["богдановна"], "богданович": ["богданович"], "богдановна": ["богданівна"], "борис": ["борис"], "борисивна": ["борисовна"], "бориславивна": ["бориславовна"], "бориславовна": ["бориславівна"], "борисович": ["борисович"], "борисовна": ["борисівна"], "бронислав": ["бронислав", "броніслав"], "брониславивна": ["брониславовна"], "брониславович": ["брониславович", "броніславович"], "брониславовна": ["броніславівна"], "ваагн": ["ваагн"], "вадим": ["вадим"], "вадимивна": ["вадимовна"], "вадимович": ["вадимович"], "вадимовна": ["вадимівна"], "валдемар": ["вальдемар"], "валентин": ["валентин"], "валентина": ["валентина"], "валентинивна": ["валентиновна"], "валентинович": ["валентинович"], "валентиновна": ["валентинівна"], "валеревич": ["валерійович"], "валеревна": ["валеріївна"], "валериан": ["валериан", "валеріан"], "валерианивна": ["валериановна"], "валериановна": ["валеріанівна"], "валериевич": ["валерійович"], "валериевна": ["валеріївна"], "валерии": ["валерий", "валерій"], "валериивна": ["валериевна", "валерьевна"], "валерииович": ["валериевич", "валерьевич"], "валерия": ["валерия", "валерія"], "ванда": ["ванда"], "варвара": ["варвара"], "васил": ["василий"], "василевич": ["васильевич", "васильович", "васільевіч"], "василевна": ["василівна", "василіївна"], "василивна": ["васильевна"], "василии": ["василий", "василь", "василій"], "василиивна": ["васильевна"], "василина": ["василиса"], "василиса": ["василина"], "василович": ["васильевич"], "вениамин": ["вениамин", "веніамін"], "вениаминивна": ["вениаминовна"], "вениаминович": ["вениаминович", "веніамінович"], "вениаминовна": ["веніамінівна"], "вера": ["віра"], "вероника": ["вероника", "вероніка"], "видаддиевич": ["видаддиевич", "відаддієвіч"], "виктор": ["виктор", "віктор"], "викторивна": ["викторовна"], "виктория": ["виктория", "вікторія"], "викторович": ["викторович", "вікторович"], "викторовна": ["викторовна", "викторівна", "вікторовна", "вікторівна"], "вилгелмович": ["вильгельмович", "вільгельмович"], "вилена": ["вилена", "вілена"], "вилям": ["вильям", "вільям"], "виолета": ["виолета", "віолета"], "вира": ["вера"], "вита": ["вита", "віта"], "виталевич": ["віталійович"], "виталевна": ["віталівна", "віталіївна"], "виталивна": ["витальевна"], "виталии": ["виталий", "віталій"], "виталиивна": ["витальевна"], "виталииович": ["витальевич"], "виталина": ["виталина", "віталіна"], "виталия": ["виталия", "віталія"], "витолдович": ["витольдович", "вітольдович"], "влада": ["влада"], "владиленович": ["владиленович"], "владимир": ["володимир", "уладзімір"], "владимирович": ["володимирович", "уладзіміравіч"], "владимировна": ["володимирівна", "володимірівна"], "владислав": ["владислав"], "владислава": ["владислава"], "владиславивна": ["владиславовна"], "владиславович": ["владиславович"], "владиславовна": ["владиславівна"], "владлена": ["владлена"], "владленивна": ["владленовна"], "владленовна": ["владленівна"], "власович": ["власович"], "володимир": ["владимир"], "володимиривна": ["владимировна"], "володимирович": ["владимирович"], "всеволод": ["всеволод"], "всеволодивна": ["всеволодовна"], "всеволодович": ["всеволодович"], "всеволодовна": ["всеволодівна"], "вячеслав": ["вячеслав"], "вячеславивна": ["вячеславовна"], "вячеславович": ["вячеславович"], "вячеславовна": ["вячеславівна"], "гавриливна": ["гавриловна"], "гаврилович": ["гаврилович"], "гавриловна": ["гаврилівна"], "галина": ["галина"], "галия": ["галия", "галія"], "ганна": ["анна"], "гариславович": ["гариславович"], "гарри": ["гарри", "гаррі"], "геворгивна": ["геворговна"], "геворговна": ["геворгівна"], "гелфира": ["гельфира", "гельфіра"], "генефа": ["генефа"], "геннадевич": ["геннадійович"], "геннадевна": ["геннадіївна"], "геннадиевич": ["геннадійович"], "геннадиевна": ["геннадіївна"], "геннадии": ["геннадий", "геннадій"], "геннадиивна": ["геннадиевна", "геннадьевна"], "геннадииович": ["геннадиевич", "геннадьевич"], "генрих": ["генрих", "генріх"], "генрихович": ["генрихович", "генріхович"], "георгиевич": ["георгиевич", "георгійович", "георгієвич"], "георгиевна": ["георгіївна"], "георгии": ["георгий", "георгій"], "георгиивна": ["георгиевна"], "георгииович": ["георгиевич"], "герман": ["герман"], "германивна": ["германовна"], "германович": ["германович"], "германовна": ["германівна"], "глеб": ["глеб", "гліб"], "глебович": ["глібович"], "глиб": ["глеб"], "глибович": ["глебович"], "гордеевич": ["гордійович"], "гордииович": ["гордеевич"], "григоревич": ["григорович", "григорійович"], "григоревна": ["григорівна", "григоріївна"], "григоривна": ["григорьевна"], "григории": ["григорий", "григорій", "грыгорый"], "григориивна": ["григорьевна"], "григорииович": ["григорьевич"], "григорович": ["григорьевич"], "давидивна": ["давидовна"], "давидовна": ["давидівна"], "даниил": ["данило"], "даниливна": ["даниловна"], "данило": ["даниил", "данило"], "данилович": ["данилович"], "даниловна": ["данилівна"], "дария": ["дария", "дарья", "дарія"], "даря": ["дарія"], "денис": ["денис"], "денисивна": ["денисовна"], "денисович": ["денисович"], "денисовна": ["денисівна"], "джамашевич": ["джамашевич"], "джонивна": ["джоновна"], "джоновна": ["джонівна"], "диана": ["диана", "діана"], "дина": ["дина", "діна"], "дмитривна": ["дмитриевна"], "дмитриевич": ["дмитрович"], "дмитриевна": ["дмитрівна", "дмитріївна"], "дмитрии": ["дмитрий", "дмитро", "дмитрій"], "дмитриивна": ["дмитриевна"], "дмитро": ["дмитрий"], "дмитрович": ["дмитриевич"], "евген": ["евгений"], "евгеневич": ["євгенович", "євгенійович", "євгенієвич"], "евгеневна": ["евгенівна", "євгенівна", "євгеніївна"], "евгенивна": ["евгеньевна"], "евгениевич": ["евгеньевич"], "евгении": ["евгений", "євген", "євгеній"], "евгениивна": ["евгеньевна"], "евгенииович": ["евгеньевич"], "евгения": ["евгения", "євгенія"], "евгенович": ["евгеньевич"], "евграфович": ["евграфович", "євграфович"], "евдокия": ["евдокия", "євдокія"], "евелина": ["евеліна", "эвелина"], "евстафевич": ["євстафійович"], "евстафевна": ["євстафіївна"], "евстафиивна": ["евстафьевна"], "евстафииович": ["евстафьевич"], "егор": ["егор", "єгор"], "егоривна": ["егоровна"], "егорович": ["егорович", "єгорович"], "егоровна": ["єгорівна"], "едуард": ["едуард", "эдуард"], "едуардивна": ["эдуардовна"], "едуардович": ["едуардович", "эдуардович"], "едуардовна": ["едуардівна"], "екатерина": ["катерина"], "елвина": ["ельвіна", "эльвина"], "елвира": ["ельвіра", "эльвира"], "елена": ["елена", "олена", "єлєна"], "елеонора": ["елеонора", "элеонора"], "елзара": ["ельзара", "эльзара"], "елизавета": ["елизавета", "єлизавета", "єлізавета"], "елина": ["еліна", "элина"], "елла": ["елла", "элла"], "еллана": ["еллана", "эллана"], "емануилович": ["эммануилович"], "емеляновна": ["омелянівна"], "емилевич": ["емільович", "емільєвич", "эмильевич"], "емилия": ["емілія", "эмилия"], "емилович": ["эмильевич"], "емма": ["емма", "эмма"], "еммануилович": ["емануілович", "емануїлович", "еммануїлович", "эммануилович"], "ервинович": ["ервінович", "эрвинович"], "ерикивна": ["эриковна"], "ериковна": ["еріківна"], "ернестивна": ["эрнестовна"], "ернестовна": ["ернестівна"], "ерофеи": ["ерофей", "єрофей"], "ефимович": ["юхимович"], "ефимовна": ["юхимівна"], "ефрем": ["ефрем", "єфрем"], "ефремович": ["ефремович", "єфремович"], "жанетта": ["жанетта"], "жанна": ["жанна"], "жаннета": ["жаннета"], "захар": ["захар"], "захарии": ["захарий", "захарій"], "захарович": ["захарович"], "зинаида": ["зинаида", "зінаїда"], "зиновевич": ["зіновійович"], "зиновевна": ["зіновіївна"], "зиновии": ["зиновий", "зіновій"], "зиновиивна": ["зиновьевна"], "зиновииович": ["зиновьевич"], "златослава": ["златослава"], "зоряна": ["зоряна"], "зоя": ["зоя"], "ибрагимович": ["ибрагимович", "ібрагимович", "ібрагімович"], "иван": ["иван", "іван"], "иванавич": ["иванович"], "иванивна": ["ивановна"], "иванна": ["иванна", "іванна"], "иванович": ["иванович", "іванавіч", "іванович"], "ивановна": ["иванівна", "іванівна"], "игор": ["игорь", "ігор"], "игоревич": ["игоревич", "ігоревич", "ігорович"], "игоревна": ["игоревна", "ігоревна", "ігорівна"], "игоривна": ["игоревна"], "игорович": ["игоревич"], "изабелла": ["изабелла", "ізабелла"], "измаил": ["измаил", "ізмаїл"], "иларионивна": ["илларионовна"], "иларионович": ["илларионович"], "илинична": ["іллівна"], "илич": ["ілліч"], "илкович": ["илькович", "ількович"], "илларионович": ["іларіонович"], "илларионовна": ["іларіонівна"], "илливна": ["ильинична"], "иллич": ["ильич"], "илля": ["илья"], "илона": ["илона", "ілона"], "иля": ["ілля"], "инга": ["инга", "інга"], "инеса": ["инесса"], "инесса": ["инесса", "інеса", "інесса"], "инна": ["инна", "інна"], "иович": ["иович", "іович"], "ион": ["ион", "іон"], "иосип": ["иосиф"], "иосипивна": ["иосифовна"], "иосипович": ["иосифович"], "иосиф": ["йосип"], "иосифович": ["йосипович"], "иосифовна": ["йосипівна"], "ираида": ["ираида", "іраїда"], "ирена": ["ирэна", "ірена"], "ирина": ["ирина", "ірина", "іріна"], "иринарх": ["иринарх", "ірінарх"], "исаакович": ["ісакович"], "исааковна": ["ісаківна"], "исакивна": ["исааковна"], "исакович": ["исаакович"], "ия": ["ия", "ія"], "камилович": ["камилович", "камілович"], "капитолина": ["капитолина", "капітоліна"], "каренович": ["каренович"], "карина": ["карина", "каріна"], "карине": ["каринэ", "каріне"], "катерина": ["екатерина"], "киазович": ["киазович", "кіазович"], "ким": ["ким", "кім"], "киндратович": ["кондратьевич"], "кира": ["кира", "кіра"], "кириливна": ["кирилловна"], "кирилл": ["кирило"], "кириллович": ["кирилович"], "кирилловна": ["кирилівна"], "кирило": ["кирилл"], "кирилович": ["кириллович"], "клавдия": ["клавдия", "клавдія"], "клара": ["клара"], "климивна": ["климовна"], "климовна": ["климівна"], "кондратевич": ["кіндратович"], "константин": ["костянтин"], "константинивна": ["константиновна"], "константинович": ["константинович", "костянтинович"], "константиновна": ["константинівна", "костянтинівна"], "костянтин": ["константин"], "костянтинивна": ["константиновна"], "костянтинович": ["константинович"], "кристина": ["кристина", "крістіна", "христина"], "ксения": ["ксения", "ксенія"], "кузмивна": ["кузьминична"], "кузминична": ["кузьмівна"], "кузмич": ["кузьмич", "кузьміч"], "курбанбаиович": ["курбанбайович"], "лаврентевич": ["лаврентійович"], "лаврентевна": ["лаврентіївна"], "лаврентиивна": ["лаврентьевна"], "лаврентииович": ["лаврентьевич"], "лада": ["лада"], "лали": ["лали", "лалі"], "лариса": ["лариса"], "лвивна": ["львовна"], "лвович": ["львович"], "лвовна": ["львівна"], "лев": ["лев"], "леонард": ["леонард"], "леонивна": ["леоновна"], "леонид": ["леонид", "леонід"], "леонидивна": ["леонидовна"], "леонидович": ["леонидович", "леонідович", "леонідовіч"], "леонидовна": ["леонидівна", "леонідівна"], "леоновна": ["леонівна"], "леонтевич": ["леонтійович"], "леонтевна": ["леонтіївна"], "леонтиевич": ["леонтійович"], "леонтиевна": ["леонтіївна"], "леонтии": ["леонтий", "леонтій"], "леонтиивна": ["леонтиевна", "леонтьевна"], "леонтииович": ["леонтиевич", "леонтьевич"], "леополдович": ["леопольдович"], "леся": ["леся"], "лиана": ["лиана", "ліана"], "лидия": ["лидия", "лідія"], "лилиана": ["лилиана", "ліліана"], "лилита": ["лилита", "ліліта"], "лилия": ["лилия", "лілія"], "лиляна": ["лиляна", "ліляна"], "лина": ["лина", "ліна"], "луарсабович": ["луарсабович"], "луиза": ["луиза", "луіза", "луїза"], "лука": ["лука"], "лукивна": ["лукинична"], "лукинична": ["луківна"], "лукич": ["лукич"], "любава": ["любава"], "любов": ["любов", "любовь"], "любомира": ["любомира"], "любомиривна": ["любомировна"], "любомирович": ["любомирович"], "любомировна": ["любомирівна"], "любославович": ["любославович"], "людвигович": ["людвигович", "людвігович"], "людмила": ["людмила"], "маия": ["майя"], "макаривна": ["макаровна"], "макарович": ["макарович"], "макаровна": ["макарівна"], "максим": ["максим"], "максимивна": ["максимовна"], "максимович": ["максимович"], "максимовна": ["максимівна"], "малвина": ["мальвина", "мальвіна"], "маликивна": ["маликовна"], "маликовна": ["маліківна"], "маргарита": ["маргарита"], "марианна": ["марианна", "маріанна"], "марина": ["марина"], "мария": ["мария", "марія"], "марк": ["марк", "марко"], "маркивна": ["марковна"], "марко": ["марк"], "маркович": ["маркович"], "марковна": ["марковна", "марківна"], "марта": ["марта"], "мартинович": ["мартинович", "мартынович"], "маруша": ["маруша"], "матвеевич": ["матвійович"], "матвеи": ["матвій"], "матвии": ["матвей"], "матвииович": ["матвеевич"], "мечиславович": ["мечиславович"], "микита": ["никита"], "микитович": ["никитович"], "микола": ["николай"], "миколаевич": ["николаевич"], "миколаи": ["николай"], "миколаивна": ["николаевна"], "миколаиович": ["николаевич"], "минас": ["минас", "мінас"], "мирон": ["мирон"], "миронивна": ["мироновна"], "миронович": ["миронович"], "мироновна": ["миронівна"], "мирослав": ["мирослав"], "мирослава": ["мирослава"], "мирославивна": ["мирославовна"], "мирославович": ["мирославович"], "мирославовна": ["мирославівна"], "митрофан": ["митрофан"], "митрофанивна": ["митрофановна"], "митрофанович": ["митрофанович"], "митрофановна": ["митрофанівна"], "михаил": ["михайло"], "михаиливна": ["михайловна"], "михаило": ["михаил"], "михаилович": ["михайлович"], "михаиловна": ["михайлівна"], "набиевич": ["набиевич", "набієвич"], "надежда": ["надія"], "надиря": ["надиря", "надіря"], "надия": ["надежда"], "назар": ["назар"], "назарович": ["назарович"], "назия": ["назия", "назія"], "нана": ["нана"], "наталия": ["наталия", "наталья", "наталія"], "наталя": ["наталья", "наталя", "наталія"], "наум": ["наум"], "наумивна": ["наумовна"], "наумович": ["наумович"], "наумовна": ["наумівна"], "нектар": ["нектар"], "нелли": ["нелли", "неллі"], "нелля": ["нелля"], "неля": ["неля"], "несторович": ["несторович"], "никита": ["микита"], "никитович": ["микитович"], "никифор": ["никифор", "нікіфор"], "никифоривна": ["никифоровна"], "никифорович": ["никифорович", "нікіфорович"], "никифоровна": ["никифорівна"], "николаевич": ["миколайович", "миколаєвич"], "николаевна": ["миколаївна"], "николаи": ["микола", "миколай"], "нина": ["нина", "ніна"], "нинел": ["нинель", "нінель"], "ниязи": ["ниязи", "ніязі"], "нодарович": ["нодарович"], "нонна": ["нонна"], "оксана": ["оксана"], "октябрина": ["октябрина"], "олга": ["ольга"], "олег": ["олег"], "олегивна": ["олеговна"], "олегович": ["олегович"], "олеговна": ["олегівна"], "олександр": ["александр"], "олександра": ["александра"], "олександривна": ["александровна"], "олександрович": ["александрович"], "олексии": ["алексей"], "олексиивна": ["алексеевна"], "олексииович": ["алексеевич"], "олена": ["елена"], "олеся": ["олеся"], "омелянивна": ["емельяновна"], "опанасивна": ["опанасовна"], "опанасовна": ["опанасівна"], "орест": ["орест"], "ореста": ["ореста"], "орестович": ["орестович"], "оскаривна": ["оскаровна"], "оскаровна": ["оскарівна"], "остапович": ["остапович"], "павел": ["павел", "павло"], "павливна": ["павловна"], "павло": ["павел"], "павлович": ["павлович"], "павловна": ["павлівна"], "пантелеимонивна": ["пантелеймоновна"], "пантелеимоновна": ["пантелеймонівна"], "петр": ["петро"], "петривна": ["петровна"], "петро": ["петр"], "петрович": ["петрович"], "петровна": ["петрівна"], "пилипивна": ["филипповна"], "пилипович": ["филиппович"], "пинхусович": ["пинхусович", "пінхусович"], "полина": ["полина", "поліна"], "прокопевна": ["прокопівна"], "прокопивна": ["прокопьевна"], "равил": ["равиль", "равіль"], "раиса": ["раиса", "раїса"], "рамзесович": ["рамзесович"], "рашид": ["рашид"], "рашидивна": ["рашидовна"], "рашидовна": ["рашидівна"], "ревазивна": ["ревазовна"], "ревазовна": ["ревазівна"], "регина": ["регина", "регіна"], "рената": ["рената"], "ризван": ["ризван", "різван"], "римма": ["римма"], "ричард": ["ричард", "річард"], "робертивна": ["робертовна"], "робертовна": ["робертівна"], "родион": ["родион", "родіон"], "родионович": ["родионович", "родіонович"], "роза": ["роза"], "роксолана": ["роксолана"], "роман": ["роман"], "романивна": ["романовна"], "романович": ["романович"], "романовна": ["романівна"], "ростислав": ["ростислав"], "ростиславивна": ["ростиславовна"], "ростиславович": ["ростиславович"], "ростиславовна": ["ростиславівна"], "рувимович": ["рувимович"], "рудолфивна": ["рудольфовна"], "рудолфовна": ["рудольфівна"], "руслан": ["руслан"], "руслана": ["руслана"], "русланивна": ["руслановна"], "русланович": ["русланович"], "руслановна": ["русланівна"], "сабина": ["сабина", "сабіна"], "савелевич": ["савелійович"], "савелевна": ["савеліївна"], "савелиевна": ["савеліївна"], "савелиивна": ["савелиевна", "савельевна"], "савелииович": ["савельевич"], "савич": ["савич"], "самира": ["самира", "саміра"], "самоилович": ["самойлович"], "самуил": ["самуил", "самуїл"], "самуилович": ["самуилович", "самуїлович"], "самуиловна": ["самуиловна", "самуіловна"], "сахнович": ["сахнович"], "светлана": ["світлана"], "свитлана": ["светлана"], "святослав": ["святослав"], "святославивна": ["святославовна"], "святославович": ["святославович"], "святославовна": ["святославівна"], "семен": ["семен"], "семенивна": ["семеновна"], "семенович": ["семенович"], "семеновна": ["семенівна"], "сергеевич": ["сергійович"], "сергеевна": ["сергіівна", "сергіївна"], "сергеи": ["сергій"], "сергии": ["сергей"], "сергиивна": ["сергеевна"], "сергииович": ["сергеевич"], "сибилла": ["сибилла", "сібілла"], "силвестрович": ["сильвестрович"], "симеонивна": ["симеоновна"], "симеоновна": ["симеонівна"], "снежана": ["снежана", "снєжана", "сніжана"], "снижана": ["снежана"], "соломонович": ["соломонович"], "софия": ["софия", "софія"], "станислав": ["станислав", "станіслав"], "станислава": ["станислава", "станіслава"], "станиславивна": ["станиславовна"], "станиславович": ["станиславович", "станіславович"], "станиславовна": ["станиславівна", "станіславівна"], "стелла": ["стелла"], "степан": ["степан"], "степанивна": ["степановна"], "степания": ["степания", "степанія"], "степанович": ["степанович"], "степановна": ["степанівна"], "стефан": ["стефан"], "стефания": ["стефания", "стефанія"], "сулико": ["сулико", "суліко"], "сусанна": ["сусанна"], "таиса": ["таиса", "таїса"], "таисия": ["таисия", "таісія", "таїсія"], "тамара": ["тамара"], "тамила": ["тамила", "таміла"], "тарас": ["тарас"], "тарасивна": ["тарасовна"], "тарасовна": ["тарасівна"], "татяна": ["тетяна"], "теодорович": ["теодорович"], "терентевна": ["терентіївна"], "терентиивна": ["терентьевна"], "тетяна": ["татьяна"], "тимофеевич": ["тимофійович"], "тимофеевна": ["тимофіївна"], "тимофеи": ["тимофій"], "тимофии": ["тимофей"], "тимофиивна": ["тимофеевна"], "тимофииович": ["тимофеевич"], "тимур": ["тимур"], "тихонович": ["тихонович"], "трифонивна": ["трифоновна"], "трифонович": ["трифонович"], "трифоновна": ["трифонівна"], "трофимович": ["трохимович"], "трофимовна": ["трохимівна"], "трохимивна": ["трофимовна"], "трохимович": ["трофимович"], "уладзимир": ["владимир"], "уладзимиравич": ["владимирович"], "уляна": ["ульяна", "уляна"], "улянович": ["ульянович", "улянович"], "фаина": ["фаина", "фаіна"], "федир": ["федор"], "федор": ["федір"], "федоривна": ["федоровна", "фёдоровна"], "федорович": ["федорович"], "федоровна": ["федорівна"], "феликс": ["феликс", "фелікс"], "феликсивна": ["феликсовна"], "феликсович": ["феликсович", "феліксович"], "феликсовна": ["феліксівна"], "феодосиевна": ["феодосіївна"], "феодосиивна": ["феодосиевна"], "филиппович": ["пилипович"], "филипповна": ["пилипівна"], "фомич": ["фомич"], "францевич": ["францевич", "францович"], "францевна": ["францівна"], "францивна": ["францевна", "францовна"], "францович": ["францевич"], "францовна": ["францівна"], "хашматулла": ["хашматулла"], "христина": ["кристина"], "чеслав": ["чеслав"], "шабанивна": ["шабановна"], "шабановна": ["шабанівна"], "шамил": ["шамиль", "шаміль"], "шамилевич": ["шамільович"], "шамилович": ["шамильевич"], "юлевич": ["юлійович"], "юлевна": ["юліївна"], "юлиан": ["юлиан", "юліан"], "юлиана": ["юлиана", "юліана"], "юлиевна": ["юліївна"], "юлии": ["юлий", "юлій"], "юлиивна": ["юлиевна", "юльевна"], "юлииович": ["юльевич"], "юлия": ["юлия", "юлія"], "юревич": ["юрійович", "юрієвич"], "юревна": ["юріївна"], "юриевич": ["юрьевич"], "юрии": ["юрий", "юрій"], "юриивна": ["юрьевна"], "юрииович": ["юрьевич"], "юхимивна": ["ефимовна"], "юхимович": ["ефимович"], "ягубович": ["ягубович"], "якив": ["яков"], "якивна": ["яковлевна"], "якимивна": ["акимовна"], "яков": ["яків"], "якович": ["яковлевич"], "яковлевич": ["якович", "яковлевич"], "яковлевна": ["яківна"], "ян": ["ян"], "яна": ["яна"], "янивна": ["яновна"], "янина": ["янина", "яніна"], "янович": ["янович"], "яновна": ["янівна"], "ярина": ["ярина"], "ярослав": ["ярослав"], "ярослава": ["ярослава"], "ярославивна": ["ярославовна"], "ярославна": ["ярославна"], "ярославович": ["ярославович"], "ярославовна": ["ярославівна"], "ясмина": ["ясмина", "ясміна"]}

FORENAMES_DATA = ["3ахар", "абдулкадеривна", "абдулкадеровна", "абзалович", "абрамивна", "абрамович", "абрамовна", "августивна", "августовна", "авдеевич", "авдииович", "авенирович", "авксентевич", "авксентевна", "авксентиивна", "авксентииович", "адам", "адамович", "аделаида", "аделя", "адиля", "азиза", "аикануш", "акимовна", "аксентевич", "аксентиевич", "аксентииович", "алберт", "албертивна", "албертович", "албертовна", "албина", "албинович", "алевтина", "алег", "алегавич", "александр", "александра", "александрович", "александровна", "алексеевич", "алексеевна", "алексеи", "алена", "алеся", "аликович", "аликовна", "алим", "алима", "алимович", "алина", "алиса", "алия", "алла", "алона", "аляксандр", "аляксандравич", "аляксеевич", "аляксеи", "аминет", "амфиан", "анаит", "ананевич", "ананииович", "анастасия", "анатол", "анатолевич", "анатолевна", "анатоливна", "анатолиевич", "анатолии", "анатолиивна", "анатолиииович", "анатолииович", "анатолииовна", "ангелина", "андреевич", "андреевна", "андреи", "андреянна", "андриана", "андрианович", "андрии", "андриивна", "андрииович", "анжела", "анжелика", "анжелина", "анна", "аннета", "анрис", "антон", "антонавич", "антонивна", "антонина", "антонович", "антоновна", "ануарбекивна", "ануарбековна", "ануфриевич", "ануш", "аншелевич", "аншелович", "арвидович", "ареставич", "арина", "аркадевич", "аркадевна", "аркадзевич", "аркадиевич", "аркадии", "аркадиивна", "аркадииович", "арнолд", "арнолдович", "арсен", "арсенивна", "арсентевич", "арсентевна", "арсентиивна", "арсентииович", "артем", "артемович", "артур", "артуривна", "артурович", "артуровна", "арцем", "асирович", "асият", "афанасевич", "афанасевна", "афанасиивна", "афанасииович", "ашурбекивна", "ашурбековна", "барисавич", "бахлулович", "богдан", "богдана", "богданивна", "богданович", "богдановна", "боголюбивна", "боголюбовна", "борис", "борисивна", "бориславивна", "бориславовна", "борисович", "борисовна", "бронислав", "брониславивна", "брониславович", "брониславовна", "ваагн", "вадим", "вадимивна", "вадимович", "вадимовна", "валдемар", "валентин", "валентина", "валентинивна", "валентинович", "валентиновна", "валеревич", "валеревна", "валериан", "валерианивна", "валериановна", "валериевич", "валериевна", "валерии", "валериивна", "валерииович", "валерикович", "валерия", "валянцин", "ванда", "варанцивна", "варанцовна", "варвара", "васил", "василевич", "василевна", "василивна", "василии", "василиивна", "василина", "василиович", "василиса", "василович", "велеонинивна", "велеониновна", "вениамин", "вениаминивна", "вениаминович", "вениаминовна", "вера", "вероника", "видаддиевич", "виктар", "виктор", "викторивна", "виктория", "викторович", "викторовна", "виктрович", "вилгелмович", "вилена", "вилиевич", "вилииович", "вилям", "виолета", "виолетта", "вира", "вита", "виталевич", "виталевна", "виталивна", "виталии", "виталиивна", "виталииович", "виталина", "виталия", "витолдович", "влада", "владиленивна", "владиленович", "владиленовна", "владимир", "владимира", "владимирович", "владимировна", "владислав", "владислава", "владиславивна", "владиславович", "владиславовна", "владлена", "владленивна", "владленовна", "власевич", "власович", "володимир", "володимира", "володимиривна", "володимирович", "всеволод", "всеволодивна", "всеволодович", "всеволодовна", "вячаслау", "вячеслав", "вячеславивна", "вячеславович", "вячеславовна", "гавриливна", "гаврилович", "гавриловна", "галина", "галия", "ганна", "гариславович", "гарри", "гарриевич", "гаррииович", "гаянивна", "гаяновна", "геворгивна", "геворговна", "гелфира", "генадиевна", "генадии", "генадиивна", "генефа", "геннадевич", "геннадевна", "геннадиевич", "геннадиевна", "геннадии", "геннадиивна", "геннадииович", "генрих", "генрихович", "георгиевич", "георгиевна", "георгии", "георгиивна", "георгииович", "герман", "германивна", "германович", "германовна", "геронтевна", "гиевич", "гииович", "гиронтиивна", "глеб", "глебович", "глиб", "глибович", "гнативна", "гнатович", "гордеевич", "гордиивна", "гордииович", "григоревич", "григоревна", "григоривна", "григории", "григориивна", "григорииович", "григорович", "гулнара", "гюлнара", "давидивна", "давидовна", "даниил", "данил", "даниливна", "данило", "данилович", "даниловна", "дария", "даря", "денис", "денисивна", "денисович", "денисовна", "джамалович", "джамашевич", "джанивна", "джановна", "джема", "джемма", "джонивна", "джоновна", "джумберивна", "джумберовна", "диана", "дина", "дмитривна", "дмитриевич", "дмитриевна", "дмитрии", "дмитриивна", "дмитро", "дмитрович", "евген", "евгеневич", "евгеневна", "евгенивна", "евгениевич", "евгении", "евгениивна", "евгенииович", "евгения", "евгенович", "евграфович", "евдокия", "евелина", "евстафевич", "евстафевна", "евстафиивна", "евстафииович", "евстахович", "евстигнеевич", "евстигнииович", "егнатевна", "егнативна", "егор", "егоривна", "егорович", "егоровна", "едуард", "едуардивна", "едуардович", "едуардовна", "екатерина", "елвина", "елвира", "елена", "елеонора", "елзара", "елизавета", "елина", "елла", "еллана", "емануилович", "емеляновна", "емилевич", "емилииович", "емилия", "емилович", "емма", "еммануилович", "епинетивна", "епинетовна", "ервинович", "еремеевич", "ерикивна", "ериковна", "ернестивна", "ернестовна", "ерофеевич", "ерофеевна", "ерофеи", "ерофиивна", "ерофииович", "ефимович", "ефимовна", "ефрем", "ефремович", "жанетта", "жанна", "жаннета", "захар", "захарии", "захарович", "зеновевич", "зеновииович", "зиедонисивна", "зиедонисовна", "зинаида", "зиновевич", "зиновевна", "зиновии", "зиновиивна", "зиновииович", "златослава", "зоряна", "зоя", "зуфаривна", "зуфаровна", "ибрагимович", "иван", "иванавич", "иванивна", "иваниивна", "иванна", "иванович", "ивановна", "игар", "игнатевич", "игнатевна", "игнатовна", "игор", "игоревич", "игоревна", "игоривна", "игорович", "игоровна", "изабелла", "измаил", "изосимивна", "изосимовна", "иларионивна", "иларионович", "илдусивна", "илдусовна", "илинична", "илич", "илкович", "илларионович", "илларионовна", "илливна", "иллич", "илля", "илона", "иля", "инга", "инеса", "инесса", "инна", "инсафивна", "инсафовна", "иович", "ион", "иосип", "иосипивна", "иосипович", "иосиф", "иосифович", "иосифовна", "иполитивна", "ипполитовна", "ираида", "ирена", "ирина", "иринарх", "исаакович", "исааковна", "исакивна", "исакович", "ия", "камилович", "канстанцинавич", "капитолина", "каренович", "карина", "карине", "карлович", "карпивна", "карповна", "катерина", "керимивна", "керимовна", "киазович", "ким", "киндратович", "кира", "кириливна", "кирилл", "кириллович", "кирилловна", "кирило", "кирилович", "клавдиевич", "клавдииович", "клавдия", "клара", "климивна", "климовна", "кондратевич", "константин", "константинивна", "константинович", "константиновна", "констянтинивна", "корнелиевич", "корнелииович", "костянтин", "костянтинивна", "костянтинович", "костятинивна", "кристина", "ксаверевич", "ксаверович", "ксения", "ксеня", "кузмивна", "кузминивна", "кузминична", "кузмич", "курбанбаиович", "лаврентевич", "лаврентевна", "лаврентиивна", "лаврентииович", "лада", "лали", "лариса", "лвивна", "лвович", "лвовна", "лев", "леонард", "леонивна", "леонид", "леонидивна", "леонидович", "леонидовна", "леоновна", "леонтевич", "леонтевна", "леонтиевич", "леонтиевна", "леонтии", "леонтиивна", "леонтииович", "леополдович", "леся", "лиана", "лидия", "лилиана", "лилита", "лилия", "лиляна", "лина", "луарсабович", "луиза", "лука", "лукивна", "лукинична", "лукич", "любава", "любов", "любомира", "любомиривна", "любомирович", "любомировна", "любославович", "людвигивна", "людвигович", "людвиговна", "людмила", "маиоривна", "маиоровна", "маия", "макаривна", "макарович", "макаровна", "максим", "максимивна", "максимович", "максимовна", "малвина", "маликивна", "маликовна", "маргарита", "марианна", "марина", "мария", "марк", "маркивна", "марко", "маркович", "марковна", "марленивна", "марленовна", "марта", "мартинивна", "мартинович", "мартиновна", "маруша", "матвеевич", "матвеи", "матвии", "матвииович", "мезенивна", "мезеновна", "методиевич", "методииович", "мечиславович", "микалаевич", "микита", "микитович", "микола", "миколаевич", "миколаи", "миколаивна", "миколаиович", "миколоивна", "минас", "мирон", "миронивна", "миронович", "мироновна", "мирослав", "мирослава", "мирославивна", "мирославович", "мирославовна", "митрофан", "митрофанивна", "митрофанович", "митрофановна", "мифодиевич", "мифодииович", "михаил", "михаилавич", "михаиливна", "михаило", "михаилович", "михаиловна", "моисеевич", "моисеевна", "мусиивна", "мусииович", "набиевич", "набииович", "надежда", "надиря", "надия", "назар", "назарович", "назия", "нана", "наталия", "наталя", "наум", "наумивна", "наумович", "наумовна", "нектар", "нелли", "нелля", "неля", "нестеровна", "несторивна", "несторович", "никита", "никитович", "никифор", "никифоривна", "никифорович", "никифоровна", "никодимивна", "никодимовна", "николаевич", "николаевна", "николаи", "нина", "нинел", "ниязи", "нодарович", "нонна", "оганесивна", "оганесовна", "оксана", "октябрина", "олга", "олег", "олегивна", "олегович", "олеговна", "олександивна", "олександр", "олександра", "олександривна", "олександрович", "олексии", "олексиивна", "олексииович", "олена", "олеся", "омелянивна", "онуфрииович", "опанасивна", "опанасович", "опанасовна", "орест", "ореста", "орестович", "оскаривна", "оскаровна", "остапович", "павел", "павливна", "павло", "павлович", "павловна", "панасович", "пантелеевич", "пантелеимонивна", "пантелеимоновна", "пантелииович", "парасковия", "парфеневич", "парфеневна", "парфениивна", "парфенович", "петр", "петривна", "петро", "петрович", "петровна", "пилипивна", "пилипович", "пинхусович", "полина", "полихрониевич", "полихронииович", "прасковя", "прокопевна", "прокопивна", "прокопииович", "прокопович", "прокофевич", "прохоривна", "прохоровна", "равил", "радиевич", "радииович", "размикивна", "размиковна", "раиса", "рамзесович", "рашид", "рашидивна", "рашидовна", "ревазивна", "ревазовна", "регина", "реджиналдивна", "реджиналдовна", "реинголдивна", "реинголдовна", "реинисивна", "реинисовна", "рената", "рефативна", "рефатовна", "ризван", "римма", "ричард", "робертивна", "робертовна", "родион", "родионович", "роза", "роксолана", "роман", "романивна", "романович", "романовна", "ростислав", "ростиславивна", "ростиславович", "ростиславовна", "рувимович", "рудолфивна", "рудолфовна", "руслан", "руслана", "русланивна", "русланович", "руслановна", "сабина", "савелевич", "савелевна", "савелиевна", "савелиивна", "савелииович", "савич", "самииливна", "самира", "самоилович", "самуил", "самуилович", "самуиловна", "сахнович", "светлана", "светозаровна", "свитлана", "свитозаривна", "святослав", "святославивна", "святославович", "святославовна", "семен", "семенивна", "семенович", "семеновна", "серанивна", "серановна", "сергеевич", "сергеевна", "сергеи", "сергии", "сергиивна", "сергииович", "сибилла", "силвестрович", "симеонивна", "симеоновна", "снежана", "снижана", "соломонович", "софия", "станислав", "станислава", "станиславивна", "станиславович", "станиславовна", "стелла", "степан", "степанивна", "степания", "степанович", "степановна", "стефан", "стефания", "сулико", "сусанна", "сяргеи", "тагибекивна", "тагибековна", "таиса", "таисия", "тамара", "тамила", "тарас", "тарасивна", "тарасовна", "татяна", "теодорович", "теофанивна", "теофановна", "терентевна", "терентиивна", "тетяна", "тимофеевич", "тимофеевна", "тимофеи", "тимофии", "тимофиивна", "тимофииович", "тимур", "тихонович", "трифонивна", "трифонович", "трифоновна", "трофимович", "трофимовна", "трохимивна", "трохимович", "уладзимир", "уладзимиравич", "уляна", "улянович", "фаина", "фархадовна", "фарходивна", "фатехивна", "фатеховна", "федаравич", "федир", "федор", "федоривна", "федорович", "федоровна", "федосеевна", "федосиивна", "феликс", "феликсивна", "феликсович", "феликсовна", "феодосиевна", "феодосиивна", "филиппович", "филипповна", "фомич", "францевич", "францевна", "францивна", "францович", "францовна", "харлампиевич", "харлампииович", "хашматулла", "христина", "цезаривна", "чеслав", "шабанивна", "шабановна", "шакиривна", "шакировна", "шамил", "шамилевич", "шамилевна", "шамилиивна", "шамилович", "ювеналевич", "ювеналиевна", "ювеналиивна", "ювеналииович", "юлевич", "юлевна", "юлиан", "юлиана", "юлианивна", "юлиевна", "юлии", "юлиивна", "юлииович", "юлия", "юревич", "юревна", "юриевич", "юрии", "юриивна", "юрииович", "юхимивна", "юхимович", "ягубович", "якив", "якивна", "якимивна", "яков", "якович", "яковлевич", "яковлевна", "ян", "яна", "янивна", "янина", "янович", "яновна", "яремович", "ярина", "ярополкивна", "ярослав", "ярослава", "ярославивна", "ярославна", "ярославович", "ярославовна", "ясмина"]

FOLD = str.maketrans({'і': 'и', 'ї': 'и', 'й': 'и', 'є': 'е', 'ё': 'е', 'э': 'е', 'ґ': 'г', 'ы': 'и', 'ў': 'у', 'ь': None, 'ъ': None, '’': None, 'ʼ': None, "'": None})

UK = re.compile('[іїєґІЇЄҐ]')

CYR = re.compile('[Ѐ-ӿ]')

LAT = re.compile('[A-Za-zÀ-ɏ]')

MOJI_PAIR = re.compile('[РС][ЂЃ‚ѓ„…†‡€‰Љ‹ЊЌЋЏђ‘’“”•–—™љ›њќћџЎўЈ¤Ґ¦§Ё©Є«¬®Ї°±Ііґµ¶·ё№є»јЅѕї]')

MOJI_CHAR = re.compile('[ЂЃ‚ѓ„…†‡€‰Љ‹ЊЌЋЏђ‘“”•™љ›њќћџЈ¤¦§©«¬®°±µ¶·№»јЅѕ\\ufffd]')

def mojibake(x):
    return bool(MOJI_CHAR.search(x)) or len(MOJI_PAIR.findall(x)) >= 2

PATR_END = re.compile('(ович|евич|євич|йович|ьович|івна|ївна|овна|евна|ична|авіч|евіч|аўна|еўна)\\Z')

INIT_SWAPS = {('м', 'н'), ('н', 'м'), ('о', 'а'), ('а', 'о')}

def fold(t):
    return t.lower().translate(FOLD)

def lev(a, b, cap=99):
    if abs(len(a) - len(b)) > cap:
        return cap + 1
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

RU_IRREG = {'михаил': ['михайлович', 'михайловна'], 'яков': ['яковлевич', 'яковлевна'], 'лев': ['львович', 'львовна'], 'павел': ['павлович', 'павловна'], 'илья': ['ильич', 'ильинична'], 'кузьма': ['кузьмич', 'кузьминична'], 'фома': ['фомич', 'фоминична'], 'лука': ['лукич', 'лукинична'], 'савва': ['саввич', 'саввична'], 'никита': ['никитич', 'никитична', 'никитович', 'никитовна'], 'петр': ['петрович', 'петровна'], 'пётр': ['петрович', 'петровна']}

UK_IRREG = {'микола': ['миколайович', 'миколаївна'], 'яків': ['якович', 'яківна'], 'лев': ['львович', 'львівна'], 'ілля': ['ілліч', 'іллівна'], 'кузьма': ['кузьмич', 'кузьмівна'], 'хома': ['хомич', 'хомівна'], 'лука': ['лукич', 'луківна'], 'сава': ['савич', 'савівна'], 'григорій': ['григорович', 'григорівна']}

BE_IRREG = {'мікалай': ['мікалаевіч', 'мікалаеўна']}

def ru_patr(g):
    g = g.lower()
    if g in RU_IRREG:
        return set(RU_IRREG[g])
    if g.endswith('ий'):
        b = g[:-2]
        return {b + 'ьевич', b + 'иевич', b + 'ьевна', b + 'иевна'}
    if g[-1] == 'й':
        return {g[:-1] + 'евич', g[:-1] + 'евна'}
    if g[-1] == 'ь':
        return {g[:-1] + 'евич', g[:-1] + 'евна'}
    if g[-1] in 'ая':
        b = g[:-1]
        return {b + 'ич', b + 'ович', b + 'ична', b + 'инична', b + 'овна'}
    if g[-1] in 'жшчщц':
        return {g + 'евич', g + 'евна', g + 'ович', g + 'овна'}
    return {g + 'ович', g + 'овна'}

def uk_patr(g):
    g = g.lower()
    if g in UK_IRREG:
        return set(UK_IRREG[g])
    if g[-1] == 'й':
        return {g + 'ович', g[:-1] + 'ївна'}
    if g[-1] == 'о':
        return {g[:-1] + 'ович', g[:-1] + 'івна'}
    if g[-1] == 'ь':
        return {g + 'ович', g[:-1] + 'івна'}
    if g[-1] in 'ая':
        b = g[:-1]
        return {b + 'ович', b + 'ич', b + 'івна'}
    out = {g + 'ович', g + 'івна'}
    if len(g) >= 4 and g[-2] == 'і':
        o = g[:-2] + 'о' + g[-1]
        out |= {o + 'ович', o + 'івна'}
    return out

def be_patr(g):
    g = g.lower()
    if g in BE_IRREG:
        return set(BE_IRREG[g])
    if g[-1] in 'йь':
        return {g[:-1] + 'евіч', g[:-1] + 'еўна'}
    if g[-1] in 'ая':
        return {g[:-1] + 'іч', g[:-1] + 'ічна'}
    return {g + 'авіч', g + 'аўна', g + 'овіч', g + 'оўна'}

ROM_ALT = {'а': ['a'], 'б': ['b'], 'в': ['v', 'w'], 'г': ['h', 'g'], 'ґ': ['g'], 'д': ['d'], 'е': ['e', 'ye', 'ie'], 'є': ['ie', 'ye', 'je', 'e'], 'ё': ['e', 'yo', 'io', 'jo', 'ye'], 'ж': ['zh', 'z', 'j'], 'з': ['z'], 'и': ['y', 'i'], 'і': ['i', 'y'], 'ї': ['i', 'yi', 'ji', 'ji'], 'й': ['i', 'y', 'j', ''], 'к': ['k'], 'л': ['l'], 'м': ['m'], 'н': ['n'], 'о': ['o'], 'п': ['p'], 'р': ['r'], 'с': ['s'], 'т': ['t'], 'у': ['u'], 'ф': ['f'], 'х': ['kh', 'h', 'ch', 'x'], 'ц': ['ts', 'c', 'tz', 'cz'], 'ч': ['ch', 'tch', 'c'], 'ш': ['sh', 's'], 'щ': ['shch', 'sch', 'shh', 'sc'], 'ь': ['', "'", 'i'], 'ъ': ['', "'"], 'ы': ['y', 'i'], 'э': ['e'], 'ю': ['iu', 'yu', 'ju'], 'я': ['ia', 'ya', 'ja'], 'ў': ['u', 'w'], "'": [''], '’': [''], 'ʼ': [''], '-': ['-', '']}

UA2010 = {'а': 'a', 'б': 'b', 'в': 'v', 'г': 'h', 'ґ': 'g', 'д': 'd', 'е': 'e', 'є': 'ie', 'ж': 'zh', 'з': 'z', 'и': 'y', 'і': 'i', 'ї': 'i', 'й': 'i', 'к': 'k', 'л': 'l', 'м': 'm', 'н': 'n', 'о': 'o', 'п': 'p', 'р': 'r', 'с': 's', 'т': 't', 'у': 'u', 'ф': 'f', 'х': 'kh', 'ц': 'ts', 'ч': 'ch', 'ш': 'sh', 'щ': 'shch', 'ь': '', 'ю': 'iu', 'я': 'ia', "'": '', '’': '', 'ʼ': '', 'ё': 'e', 'ы': 'y', 'э': 'e', 'ъ': '', 'ў': 'u', '-': '-'}

UA2010_INIT = {'є': 'ye', 'ї': 'yi', 'й': 'y', 'ю': 'yu', 'я': 'ya'}

BGN = {'а': 'a', 'б': 'b', 'в': 'v', 'г': 'g', 'д': 'd', 'е': 'e', 'ё': 'e', 'ж': 'zh', 'з': 'z', 'и': 'i', 'й': 'y', 'к': 'k', 'л': 'l', 'м': 'm', 'н': 'n', 'о': 'o', 'п': 'p', 'р': 'r', 'с': 's', 'т': 't', 'у': 'u', 'ф': 'f', 'х': 'kh', 'ц': 'ts', 'ч': 'ch', 'ш': 'sh', 'щ': 'shch', 'ъ': '', 'ы': 'y', 'ь': '', 'э': 'e', 'ю': 'yu', 'я': 'ya', 'і': 'i', 'ї': 'yi', 'є': 'ye', 'ґ': 'g', 'ў': 'w', "'": '', '’': '', 'ʼ': '', '-': '-'}

def romanize(word, system):
    w = word.lower()
    if system == 'ua2010':
        out = [UA2010_INIT.get(w[0], UA2010.get(w[0], w[0]))] + [UA2010.get(c, c) for c in w[1:]]
        return ''.join(out).replace('zgh', 'zgh')
    return ''.join((BGN.get(c, c) for c in w))

_rx_cache = {}

def rom_regex(word):
    w = word.lower()
    if w not in _rx_cache:
        _rx_cache[w] = re.compile(''.join(('(?:' + '|'.join((re.escape(a) for a in sorted(ROM_ALT.get(c, [c]), key=len, reverse=True))) + ')' for c in w)) + '\\Z')
    return _rx_cache[w]

def rom_match(cyr, lat, tol=1):
    lat = lat.lower()
    if rom_regex(cyr).match(lat):
        return True
    if tol and len(lat) >= 4:
        return any((lev(romanize(cyr, s), lat, tol) <= tol for s in ('ua2010', 'bgn')))
    return False

def init_compat(cyr_letter, lat):
    lat = lat.lower()
    return any((a and (lat.startswith(a) or (len(lat) <= 2 and a.startswith(lat))) for a in ROM_ALT.get(cyr_letter.lower(), [cyr_letter.lower()])))

def deaccent(s):
    return ''.join((c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c)))

def parts(name):
    s = re.sub("['’ʼ`]", '', unicodedata.normalize('NFC', name or ''))
    return [t.strip('-') for t in re.split('[^\\w-]+', s) if t.strip('-') and (not t.strip('-').isdigit())]

CYR2LAT = {'а': 'a', 'е': 'e', 'о': 'o', 'р': 'p', 'с': 'c', 'у': 'y', 'х': 'x', 'і': 'i', 'ј': 'j', 'ѕ': 's', 'ё': 'ë', 'ї': 'ï', 'А': 'A', 'В': 'B', 'Е': 'E', 'К': 'K', 'М': 'M', 'Н': 'H', 'О': 'O', 'Р': 'P', 'С': 'C', 'Т': 'T', 'Х': 'X', 'І': 'I', 'Ј': 'J', 'Ѕ': 'S', 'Ї': 'Ï'}

LAT2CYR = {v: k for k, v in CYR2LAT.items()}

def unhomoglyph(name):
    if not name or not (CYR.search(name) and LAT.search(name)):
        return name
    u_lat = any((LAT.match(c) and c not in LAT2CYR for c in name))
    u_cyr = any((CYR.match(c) and c not in CYR2LAT for c in name))
    if u_lat == u_cyr:
        return name
    out = ''.join((CYR2LAT.get(c, c) for c in name)) if u_lat else ''.join((LAT2CYR.get(c, c) for c in name))
    return out if not (CYR.search(out) and LAT.search(out)) else name

def script(name):
    c, l = (bool(CYR.search(name or '')), bool(LAT.search(name or '')))
    return 'cyr' if c and (not l) else 'lat' if l and (not c) else 'mixed' if c and l else 'none'

def lang(name):
    s = script(name)
    if s != 'cyr':
        return s
    if 'ў' in name.lower():
        return 'be'
    return 'uk' if UK.search(name) else 'ru'

def surname_ok_cyr(s, t):
    fs, ft = (fold(s), fold(t))
    if fs == ft:
        return True
    short = min(len(fs), len(ft))
    if short < 3:
        return False
    return lev(fs, ft, 2) <= (2 if short >= 5 else 1)

def surname_ok_lat(cyr, lat):
    return len(cyr) >= 2 and len(lat) >= 2 and rom_match(cyr, lat, 1 if len(lat) >= 5 else 0)

def surname_slots(raw, ps):
    if ',' in raw:
        head = parts(raw.split(',')[0])
        if len(head) == 1:
            return [0]
    full = [i for i, x in enumerate(ps) if len(x) >= 2]
    return full

def same_person(nt, a, b, uk_ctx):
    if not a or not b:
        return (False, 'blank')
    if mojibake(a) or mojibake(b):
        return (False, 'mojibake')
    sa, sb = (script(a), script(b))
    if sa == 'lat' and sb == 'cyr':
        a, b, sa, sb = (b, a, sb, sa)
    if sa != 'cyr' or sb not in ('cyr', 'lat'):
        return (False, f'script {sa}/{sb}')
    pa = parts(a)
    pb = parts(b) if sb == 'cyr' else [deaccent(x).lower().replace('x', 'ks') for x in parts(b)]
    if not (2 <= len(pa) <= 4 and 2 <= len(pb) <= 4) or abs(len(pa) - len(pb)) > 1 or (len(pa) != len(pb) and min(len(pa), len(pb)) != 2):
        return (False, 'part count')
    ia, ib = (surname_slots(a, pa), surname_slots(b, pb))
    for i in ia:
        s = pa[i]
        for j in ib:
            t = pb[j]
            if sb == 'cyr':
                ok = surname_ok_cyr(s, t) and (fold(s) == fold(t) or not (nt.is_forename(s) or nt.is_forename(t)))
            else:
                ok = rom_match(s, t, 0) if nt.is_forename(s) else surname_ok_lat(s, t)
            if not ok:
                continue
            ra, rb = (pa[:i] + pa[i + 1:], pb[:j] + pb[j + 1:])
            short = '' if len(ra) == len(rb) else '_nopatr'
            if sb == 'cyr':
                if all((nt.part_ok(x, y, uk_ctx) for x, y in zip(ra, rb))):
                    return (True, 'cyr' + short)
            elif all((nt.lat_part_ok(x, y) for x, y in zip(ra, rb))):
                return (True, 'lat' + short)
    return (False, 'parts')

class NameTest:

    def __init__(self):
        self.map = {k: set(v) for k, v in MAP_DATA.items()}
        self.raw = {k: set(v) for k, v in RAW_DATA.items()}
        self.forenames = set(FORENAMES_DATA)
        self.init_swaps = True

    def is_forename(self, x):
        f = fold(x)
        return f in self.forenames or bool(PATR_END.search(x.lower()))

    def part_ok(self, x, y, uk_ctx):
        fx, fy = (fold(x), fold(y))
        if not fx or not fy:
            return False
        if len(fx) > 1 and len(fy) > 1:
            return fx == fy or fy in self.map.get(fx, ())
        if len(fx) == 1 and len(fy) == 1:
            return fx == fy or (self.init_swaps and uk_ctx and ((fx, fy) in INIT_SWAPS))
        i, full = (fx, fy) if len(fx) == 1 else (fy, fx)
        return full[0] == i or any((f[0] == i for f in self.map.get(full, ())))

    def lat_part_ok(self, cyr, lat):
        forms = {f for f in {cyr.lower()} | {r.lower() for r in self.raw.get(fold(cyr), ())} if f}
        if not lat or not forms:
            return False
        if len(cyr) == 1 or len(lat) <= 2:
            return any((init_compat(f[0], lat) for f in forms))
        return any((rom_match(f, lat, 1) for f in forms))

HOMOGLYPHS = True

MAX_GROUP = 4

class Layout:

    def __init__(self, nt=None):
        self.nt = nt or NameTest()
        self._cache = {}

    def ok(self, names, i, j, uk_ctx):
        key = (i, j)
        if key not in self._cache:
            a, b = (names[i], names[j])
            if a == b and script(a) in ('cyr', 'lat'):
                self._cache[key] = (True, 'identical')
            else:
                self._cache[key] = same_person(self.nt, a, b, uk_ctx)
        return self._cache[key][0]

    def group_ok(self, names, g, uk_ctx):
        cy = [x for x in g if script(names[x]) == 'cyr']
        la = [x for x in g if script(names[x]) != 'cyr']
        if not cy:
            return False
        if not all((self.ok(names, x, y, uk_ctx) for x, y in itertools.combinations(cy, 2))):
            return False
        return all((any((self.ok(names, c, l, uk_ctx) for c in cy)) for l in la))

    def classify(self, names):
        self._cache = {}
        n = len(names)
        fixed = [unhomoglyph(x) for x in names] if HOMOGLYPHS else list(names)
        hg = fixed != list(names)
        names = fixed
        scripts = [script(x) for x in names]
        if n < 2 or 'cyr' not in scripts:
            return {'status': 'rejected', 'why': 'no cyrillic'}
        if any((s not in ('cyr', 'lat') for s in scripts)):
            return {'status': 'rejected', 'why': 'other script / blank'}
        uk_ctx = any((lang(x) in ('uk', 'be') for x in names))
        found = []
        for m in (2, 3, 4):
            if n % m:
                continue
            k = n // m
            persons = [[p + c * k for c in range(m)] for p in range(k)]
            if all((self.group_ok(names, g, uk_ctx) for g in persons)):
                found.append((f'blocks{m}', persons))
        gp = self._groups(names, 0, uk_ctx, {})
        if gp is not None and (not any((sorted(map(tuple, gp)) == sorted(map(tuple, f[1])) for f in found))):
            found.append((f'groups', gp))
        if not found:
            return {'status': 'rejected', 'why': self._why(names, uk_ctx)}
        if len(found) > 1:
            found.sort(key=lambda f: len(f[1]))
            coarse = {x: i for i, g in enumerate(found[0][1]) for x in g}
            if not all((len({coarse[x] for x in g}) == 1 for f in found[1:] for g in f[1])):
                return {'status': 'ambiguous', 'why': ','.join((f[0] for f in found))}
        lay, persons = found[0]
        if all((names[x] == names[g[0]] for g in persons for x in g)):
            return {'status': 'identical', 'layout': lay, 'persons': persons}
        if not any((len({lang(names[x]) for x in g}) > 1 for g in persons)):
            return {'status': 'same_language', 'layout': lay, 'persons': persons}
        return {'status': 'accepted', 'layout': lay, 'persons': persons, 'uk_ctx': uk_ctx, 'homoglyph': hg, 'langs': '+'.join(sorted({lang(names[x]) for x in range(n)}))}

    def _groups(self, names, i, uk_ctx, memo):
        n = len(names)
        if i == n:
            return []
        if i in memo:
            return memo[i]
        memo[i] = None
        for size in range(MAX_GROUP, 1, -1):
            if i + size > n:
                continue
            g = list(range(i, i + size))
            if self.group_ok(names, g, uk_ctx):
                rest = self._groups(names, i + size, uk_ctx, memo)
                if rest is not None:
                    memo[i] = [g] + rest
                    break
        return memo[i]

    def _why(self, names, uk_ctx):
        n = len(names)
        for m in (2, 3):
            if n % m == 0:
                k = n // m
                for p in range(k):
                    for c in range(1, m):
                        if not self.ok(names, p, p + c * k, uk_ctx):
                            return f'blocks{m}: seat {p} vs {p + c * k}: {self._cache[p, p + c * k][1]}'
        return 'no layout' + (' (odd n)' if n % 2 and n % 3 else '')

def language_copy_targets(names, orcids):
    if names is None or len(names) < 2 or len(names) > 200:
        return None
    names = [x or "" for x in names]
    n = len(names)
    orcids = list(orcids) if orcids is not None and len(orcids) == n else [None] * n
    nt = getattr(sys, "_oxjob2186_name_test", None)
    if nt is None:
        nt = NameTest()
        sys._oxjob2186_name_test = nt
    c = Layout(nt).classify(names)
    if c["status"] != "accepted":
        return None
    t = list(range(n))
    for g in c["persons"]:
        if len({o for o in (orcids[x] for x in g) if o}) > 1:
            return None
        for x in g:
            t[x] = g[0]
    return t

return language_copy_targets(names, orcids)
$$


In [0]:
-- oxjob #2186 step 5b-F: per work whose byline is language copies of the same people in the same order (Ukrainian /
-- Russian / Belarusian / romanized copies: 'Кулик, Ірина Володимирівна' / 'Кулик, Ирина Владимировна' / 'Kulyk, Iryna V.'),
-- the seat each seat folds into (t, 0-based, itself when kept) and a hash of the byline's names (h). CreateWorkAuthorships
-- step 5b-F applies t only when the names still hash to h (printed-name guard: a byline that changed since is left alone).
-- Candidates (a loose prefilter; the rule itself is the UDF): 2-200 seats, a Cyrillic seat, and seat 0's surname has a
-- look-alike (coarse Latin skeleton, edit distance <= 2) at seat 1, n/2, n/3 or n/4: ~813K works (13 Nov 2026), ~402K rows.
-- Bar (oxjobs #2186 EXPLORE 'uk/ru layout rule'): Opus, blind, 200/200 judged right (120 random + 80 homoglyph additions).
-- Revert: drop this task (jobs/walden_end2end.yaml), this notebook and the #2186 5b-F edits in CreateWorkAuthorships.
CREATE OR REPLACE TABLE openalex.works.work_byline_language_copies
COMMENT 'oxjob #2186 step 5b-F: per language-copy byline, target seat per seat (t) and xxhash64 of the names (h). Rebuilt each run by Language_Copies.'
AS
WITH b AS (
  SELECT id AS work_id, size(authorships) AS n,
         transform(authorships, a -> a.raw_author_name) AS names,
         transform(authorships, a -> a.raw_orcid) AS orcids,
         transform(authorships, a -> filter(split(regexp_replace(translate(translate(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(lower(coalesce(a.raw_author_name, '')), 'shch|sch|shh', 's'), 'sh', 's'), 'ch', 'c'), 'zh', 'z'), 'kh', 'h'), 'ts|tz|cz', 'c'), 'ya|ia|ja', 'a'), 'yu|iu|ju', 'u'), 'ye|ie|je', 'e'), 'yi|ji', 'i'), '[yj]', 'i'), 'x', 'ks'), 'w', 'v'), 'g', 'h'), 'абвгґдеєёжзиіїйклмнопрстуфхцчшщыэюяўьъ', 'abvhhdeeezziiiiklmnoprstufhccssieuau'), 'áàâäãåéèêëíìîïóòôöõúùûüçñšžčćłńśźżőű', 'aaaaaaeeeeiiiiooooouuuucnszcclnszzou'), '[^a-z]+', ' '), ' '), t -> length(t) >= 3)) AS sk
  FROM openalex.works.openalex_works_base
  WHERE size(authorships) BETWEEN 2 AND 200
    AND exists(authorships, a -> a.raw_author_name RLIKE '\\p{IsCyrillic}')
),
c AS (
  SELECT work_id, names, orcids FROM b
  WHERE exists(sk[0], t -> exists(sk[1], u -> levenshtein(t, u) <= 2))
     OR (n % 2 = 0 AND exists(sk[0], t -> exists(sk[n DIV 2], u -> levenshtein(t, u) <= 2)))
     OR (n % 3 = 0 AND exists(sk[0], t -> exists(sk[n DIV 3], u -> levenshtein(t, u) <= 2)))
     OR (n % 4 = 0 AND exists(sk[0], t -> exists(sk[n DIV 4], u -> levenshtein(t, u) <= 2)))
),
r AS (
  SELECT work_id, xxhash64(names) AS h, openalex.works.language_copy_targets(names, orcids) AS t FROM c
)
SELECT work_id, h, t FROM r WHERE t IS NOT NULL
